In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
import os

base_dir = os.path.dirname(os.getcwd()) 
file_path = os.path.join(base_dir, 'data', 'raw', 'online_retail.csv')

df = pd.read_csv(file_path)

print(f"Original Shape: {df.shape}")

Original Shape: (541909, 8)


In [10]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [7]:
# We cannot track user behavior or make recommendations without a CustomerID.
# We also drop rows where Description is missing.
df = df.dropna(subset=['CustomerID', 'Description'])

df.shape

(406829, 8)

In [12]:
# In this dataset, Invoice numbers starting with 'C' are cancellations/returns.
# For a "Cart Abandonment" predictor, we usually only want successful attempts 
# (or pure browsing). we remove cancelled orders for simplicity.
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

df.shape

(397924, 8)

In [14]:
# Quantity or UnitPrice less than or equal to 0 usually indicates errors 
# or adjustments, not actual shopping behavior.
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

df.shape

(397884, 8)

In [16]:
# Convert InvoiceDate to datetime objects (crucial for the Timeline feature)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Convert CustomerID to integer (it often loads as a float like 12348.0)
df['CustomerID'] = df['CustomerID'].astype(int)

df.shape

(397884, 8)

In [18]:
# CREATE USEFUL FEATURES (Feature Engineering)
# Calculate Total Spend per line item
df['Total_Spend'] = df['Quantity'] * df['UnitPrice']

df.shape

(397884, 9)

In [20]:
# Sometimes users double-click or data logs duplicate entries. Remove duplicates.
df = df.drop_duplicates()

df.shape

(392692, 9)

In [22]:
# Save the cleaned data to CSV
df.to_csv('../data/processed/clean_data.csv', index=False)

print(f"Cleaned Shape: {df.shape}")
print("Clean data saved to: data/processed/clean_data.csv")

Cleaned Shape: (392692, 9)
Clean data saved to: data/processed/clean_data.csv


In [24]:
# Compare row counts
print("Original rows likely had ~541,000.")
print(f"The clean dataset has {df.shape[0]} rows.")
print(f"We removed {541909 - df.shape[0]} garbage rows.")

Original rows likely had ~541,000.
The clean dataset has 392692 rows.
We removed 149217 garbage rows.
